# Bedding × Dinomaly — inference tutorial

End-to-end demo of running the 6-channel ALL6 high-res (672×672) Dinomaly pipeline on the bedding val set.

Covers:
1. Loading the saved pipeline (YAML + .pt).
2. Reading a cu3s cube and feeding it to the pipeline.
3. Visualising the VIS / SWIR triplets + score heatmap + GT mask overlay (the 6-channel analog of the lentils notebook's RGB/CIR/custom triptych).
4. Applying the verified-lossless inference speedup recipe (TF32 + bf16 autocast + `torch.compile`) — ~3× faster, AUROC preserved to 5 decimals.
5. Headline metric table (Dinomaly vs EAD baseline) + per-class pixel AUROC bar.
6. ROC curves + confusion matrices.
7. Cross-method comparison (where ALL6 wins / where it trails EAD).
8. Outstanding investigations checklist.

Companion notebook: `bedding_all6_train_tutorial.ipynb` (build + train the pipeline from scratch). This notebook covers everything else — running the trained pipeline, qualitative inspection, the lossless speedup recipe, and the headline + per-class results.

> **Prerequisites**
>
> 1. Run from `/home/dev/anish/cuvis-ai-dinomaly`.
> 2. Activate the repo's env: `source .venv/bin/activate && jupyter lab`.
> 3. The pretrained pipeline YAML + PT must exist at `/mnt/data/cuvis_ai_outputs/dinomaly_bedding_all6_highres_30ep/trained_models/` (produced by the production 20-ep pilot). Train your own with `bedding_all6_train_tutorial.ipynb` if needed.
> 4. The bedding val cu3s files and NPZ exports must be on the same machine. The dataset is published at `cubert-gmbh/X4_SWIR_Industrial_Foreign_Object_Detection_Bedding` — `utils.load_bedding_cu3s_path` will absorb the swap in one line (uncomment the HF branch).
> 5. GPU strongly recommended for the speedup-recipe section (TF32 + bf16 autocast + `torch.compile` are GPU-only optimisations).


> **HuggingFace readiness.** This tutorial currently reads cu3s files from the local mount at `/mnt/data/bedding_dataset/exported/val/`. The dataset is published at `cubert-gmbh/X4_SWIR_Industrial_Foreign_Object_Detection_Bedding`; the loader (`utils.load_bedding_cu3s_path`) can `huggingface_hub.snapshot_download(...)` transparently — the notebook stays unchanged.


In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cuvis
import numpy as np
import torch

# Local helpers (utils.py sits next to this notebook).
sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    resolve_default_config, load_bedding_cu3s_path,
    render_input_triplets, render_inference_panel,
    apply_lossless_speedups,
)

from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.utils.node_registry import NodeRegistry
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.execution import Context

config = resolve_default_config()
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
print('Device:', device)
print('Pipeline YAML:', config['pipeline_yaml'])
print('Pipeline PT  :', config['pipeline_pt'])

## 1 · Load the saved pipeline

We load the high-res 20-epoch ALL6 model. The pipeline graph is: `LentilsAnomalyDataNode` → `MinMaxNormalizer` → `FixedHyperspectralSelector(target_wavelengths=(625,550,450,1450,1200,1050))` → `DinomalyDetector(input_channels=6, image_size=672)` → `decider`.

In [ ]:
registry = NodeRegistry()
registry.load_plugins(str(config['plugins_yaml']))

pipeline = CuvisPipeline.load_pipeline(
    config['pipeline_yaml'],
    weights_path=str(config['pipeline_pt']),
    device=str(device),
    strict_weight_loading=False,
    node_registry=registry,
)
pipeline.torch_layers.eval()
print('Nodes:', [n.name for n in pipeline.nodes])

## 2 · Run inference on a curated val frame

We hand-pick a `_nok_nok_` frame — both sides anomalous — because it gives the clearest qualitative story (multiple foreign objects, easy to see what the model finds). Other interesting frames to try (swap into `FRAME_STEM` below):

- `20250310_153943_frame_121_nok_nok_rdx_rwx` — top TP (score 0.759, multiple PLA pieces + leaf)
- `20250310_084228_frame_30_ok_ok_rd4_rw8` — normal, the `rd4_rw8` sub-recipe that the 672² model correctly rejects
- `20250311_101035_frame_10_nok_ok_rdx_rwx` — edge case: model detects, but GT mask is empty (annotation gap; see `comparisons/ead_baseline/discrepancy_investigation.md`)
- `20250311_101004_frame_9_nok_ok_rdx_rwx` — near-miss FN (just below best-F1 threshold)

In [ ]:
FRAME_STEM = '20250310_153943_frame_121_nok_nok_rdx_rwx'
cu3s_path = load_bedding_cu3s_path(FRAME_STEM, val_root=config['cu3s_val_root'])
assert cu3s_path.is_file(), cu3s_path

# Read the 6-band cube straight from cu3s, mirroring the EfficientAD example's inference pattern.
mesu = cuvis.SessionFile(str(cu3s_path)).get_measurement(0)
cube_hwc = mesu.data['cube'].array.astype(np.float32)
wavelengths_nm = np.asarray(mesu.data['cube'].wavelength, dtype=np.int32)
print(f'cube shape: {cube_hwc.shape}, dtype: {cube_hwc.dtype}, wavelengths: {wavelengths_nm.tolist()}')

In [ ]:
# Visualise the two 3-channel triplets the pipeline sees after the selector reorders into
# (625, 550, 450, 1450, 1200, 1050) nm — VIS-RGB on the left, SWIR-pseudo-RGB on the right.
# Note: the raw cube here is in its native acquisition order; the selector inside the pipeline
# does the reorder. For *display* we replicate that order on the raw cube.

ASC_TO_TARGET_ORDER = []
for nm in (625, 550, 450, 1450, 1200, 1050):
    # Pick the nearest cube band for each target wavelength.
    ASC_TO_TARGET_ORDER.append(int(np.argmin(np.abs(wavelengths_nm - nm))))
cube_ordered = cube_hwc[..., ASC_TO_TARGET_ORDER][None, ...]  # [1, H, W, 6]
fig = render_input_triplets(cube_ordered, title=f'{FRAME_STEM} — VIS (625/550/450 nm) vs SWIR (1450/1200/1050 nm)')
fig.savefig(Path('outputs') / f'{FRAME_STEM}_input_triplets.png', dpi=120, bbox_inches='tight')

In [ ]:
# Build the pipeline input batch from the saved NPZ (correct shapes + dtypes).
import numpy as np_ ; import pandas as pd_
splits = pd_.read_csv(str(config['splits_csv']))
row = splits[splits['npz_path'].str.contains(FRAME_STEM, regex=False)]
assert len(row) > 0, f'{FRAME_STEM} not in splits CSV'
with np_.load(row.iloc[0]['npz_path']) as z:
    cube_np = z['cube'].astype(np_.float32)
    mask_np = z['mask'].astype(np_.int32) if 'mask' in z.files else np_.zeros(cube_np.shape[:2], dtype=np_.int32)
    wl_np   = z['wavelengths'].astype(np_.int32)
    cls_np  = z['class_mask'].astype(np_.uint8) if 'class_mask' in z.files else np_.zeros(cube_np.shape[:2], dtype=np_.uint8)
cube_hwc = cube_np ; wavelengths_nm = wl_np
batch = {
    'cube':       torch.from_numpy(cube_np[None]).to(device),
    'mask':       torch.from_numpy(mask_np[None]).to(device),
    'wavelengths': torch.from_numpy(wl_np[None]).to(device),
    'mesu_index': torch.zeros(1, dtype=torch.int64).to(device),
    'class_mask': torch.from_numpy(cls_np[None]).to(device),
}
ctx = Context(stage=ExecutionStage.INFERENCE, epoch=0, batch_idx=0, global_step=0)
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.perf_counter()
with torch.inference_mode():
    out = pipeline.forward(batch=batch, context=ctx)
torch.cuda.synchronize() if device.type == 'cuda' else None
dt_ms = (time.perf_counter() - t0) * 1000.0
score_map   = out[('dinomaly_detector', 'scores')].detach().float().cpu().numpy()
image_score = float(out[('dinomaly_detector', 'anomaly_score')].item())
print(f'fp32 baseline: {dt_ms:.1f} ms/frame  |  image score = {image_score:.4f}')


In [ ]:
# Render the qualitative panel: VIS, SWIR, score heatmap (+ GT contour if available).
MASK_PNG = config['mask_root'] / f'{FRAME_STEM}_mask.png'
gt_mask = None
if MASK_PNG.is_file():
    import cv2
    gt_mask = cv2.imread(str(MASK_PNG), cv2.IMREAD_GRAYSCALE)
    if gt_mask is not None and gt_mask.shape != score_map.shape[1:3]:
        gt_mask = cv2.resize(gt_mask, (score_map.shape[2], score_map.shape[1]),
                             interpolation=cv2.INTER_NEAREST)
fig = render_inference_panel(cube_ordered, score_map, gt_mask=gt_mask,
                              title=f'{FRAME_STEM}  ·  image score = {image_score:.4f}')
fig.savefig(Path('outputs') / f'{FRAME_STEM}_inference_panel.png', dpi=120, bbox_inches='tight')

## 3 · Apply the lossless inference speedup recipe

Three orthogonal optimisations, each verified to preserve metrics on the full val set (pixel AUROC matches fp32 to 5 decimals; image AUROC within ±0.003):

1. `torch.set_float32_matmul_precision('high')` — TF32 matmul on Ampere.
2. `torch.autocast(dtype=torch.bfloat16)` — bf16-mixed for the heavy attention layers.
3. `torch.compile(mode='reduce-overhead')` on the underlying DINOv2 model.

Combined recipe: ~3× faster end-to-end (145 ms → 52 ms/frame on RTX A4000) with no measurable accuracy drift.

In [ ]:
# Apply speedup recipe and measure the new latency.
autocast_ctx = apply_lossless_speedups(pipeline)

# First call after torch.compile is the JIT — discard, then time.
with torch.inference_mode(), autocast_ctx:
    _ = pipeline.forward(batch=batch, context=ctx)
torch.cuda.synchronize() if device.type == 'cuda' else None

samples = []
with torch.inference_mode(), autocast_ctx:
    for _ in range(5):
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        out2 = pipeline.forward(batch=batch, context=ctx)
        if device.type == 'cuda': torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)
score_map2 = out2[('dinomaly_detector', 'scores')].detach().float().cpu().numpy()
image_score2 = float(out2[('dinomaly_detector', 'anomaly_score')].item())
print(f'after speedup recipe: {np.mean(samples):.1f} ± {np.std(samples):.1f} ms/frame ({len(samples)} timed)')
print(f'image score: fp32={image_score:.5f}  optimized={image_score2:.5f}  (delta={image_score2-image_score:+.5f})')
print(f'score-map correlation vs fp32: {np.corrcoef(score_map.ravel(), score_map2.ravel())[0,1]:.6f}')

## 4 · Headline metrics

Loaded from `eval_val/report.json`. These are the val-set numbers the pilot produced.
Note: the cross-method comparison numbers (vs EAD, vs RGB pilots) live in `comparisons/headline_matrix.md` — we surface the key ones here.

In [ ]:
report = load_headline_report(eval_dir / 'report.json')
# Headline table
print(f"{'Metric':<35} {'Value':>12}")
print('-' * 50)
for k in ('pixel_auroc', 'image_auroc', 'image_auroc_p99_9', 'dice_optimal_f1_raw',
          'dice_threshold_raw', 'mean_per_class_auroc'):
    if k in report:
        v = report[k]
        print(f'{k:<35} {v:>12.4f}' if isinstance(v, (int, float)) else f'{k:<35} {v!s:>12}')

# Vs EAD baseline (numbers from comparisons/headline_matrix.md — these are pinned)
ead_baseline = {
    'pixel_auroc': 0.910,
    'image_auroc_mask_labels': 0.846,
    'image_auroc_filename_labels': 0.9396,
    'dice_optimal_f1': 0.679,
    'mean_per_class_auroc': 0.914,
}
print()
print('EAD baseline (reproduction):')
for k, v in ead_baseline.items():
    print(f'  {k:<33} {v:>12.4f}')

## 5 · Per-class pixel AUROC

23 classes, computed using EAD's methodology (per-frame normalize + background drawn from frames where the class is present). Sorted ascending so the weak classes are highlighted at the bottom.

In [ ]:
per_class_json = list(eval_dir.glob('per_class_auroc_*.json'))
if per_class_json:
    fig = plot_per_class_auroc_bar(per_class_json[0],
                                   title='Per-class pixel AUROC — ALL6 high-res (672, 20 ep)')
    fig.savefig(Path('outputs') / 'per_class_auroc_bar.png', dpi=120, bbox_inches='tight')
else:
    print('No per_class_auroc_*.json found. Run recompute_per_class_auroc.py first.')

## 6 · ROC curves + confusion matrices

Pre-rendered PNGs from `eval_val/` and `comparisons/confusion_matrices/`. We just display them inline.

In [ ]:
CM_DIR = Path('/home/dev/anish/cuvis-ai-cookbook/examples/bedding_dinomaly/comparisons/confusion_matrices')
tiles = [
    ('Pixel ROC (overall)', eval_dir / 'roc_pixel.png'),
    ('Per-class ROC', eval_dir / 'roc_per_class.png'),
    ('Pixel CM @ best-F1 thr', CM_DIR / 'all6_highres_20ep_pixel_cm.png'),
    ('Image CM (filename labels, EAD-style)', CM_DIR / 'all6_highres_20ep_image_best_F1_filename.png'),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (title, path) in zip(axes.flat, tiles):
    if path.is_file():
        ax.imshow(mpimg.imread(str(path)))
        ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f'(missing)\n{path}', ha='center', va='center')
    ax.axis('off')
fig.tight_layout()
fig.savefig(Path('outputs') / 'results_grid.png', dpi=110, bbox_inches='tight')

## 7 · Cross-method comparison

Big-picture story across the four pilots in `comparisons/headline_matrix.md`. ALL6 high-res beats EAD on every aggregate metric except Dice; the remaining Dice gap is a precision problem (FP pool at the global threshold) not a signal problem.

Where ALL6 high-res wins big over EAD reproduction (per-class pixel AUROC):

- PLA_black_16mm: +0.420
- transparent_plastic: +0.344
- alcohol: +0.123
- water&alcohol-tray: +0.082
- mean across all 23 classes: +0.059

Where ALL6 still trails EAD (these are the open work items):

- **PLA_white_2mm**: 0.835 (vs EAD 0.940). White-on-white visible-light failure mode; aspect-preserving pilots (434×1036, 504×1204) target this directly.
- **PLA_blue_2mm**: 0.908 (vs EAD 0.972). Mid-size blue plastics — similar small-object hypothesis.
- **Dice**: 0.615 (vs EAD 0.679). Likely closes via morphological cleanup of the FP pool at the global threshold — no retraining needed.

## 8 · Outstanding investigations

From `comparisons/README.md` — re-stated here so the notebook is a self-contained portal:

- [ ] Re-annotate `frame_10_nok_ok_rdx_rwx` (the one frame whose annotation JSON has zero polygons, so its mask PNG was never generated). Closes the 0.85 → 0.94 mask-vs-filename label gap permanently.
- [ ] Aspect-preserving pilots (434×1036, 504×1204) — currently training; results land in this notebook's input dir once done.
- [ ] CIR (450 / 650 / 1050 nm) + SWIR-only (1050 / 1200 / 1450 nm) ablations — falsifies whether ALL6's lift comes from SWIR alone or from VIS+SWIR fusion.
- [ ] Morphological-opening cleanup on score map — likely closes most of the remaining Dice gap to EAD.
- [ ] Identify the 2 FN frames at the best image-F1 threshold (which classes do the misses contain?).
- [ ] Curate qualitative-story sample images — narrative around the 8 attached frames on the Jira ticket.
- [ ] Upstream generalisation of `cuvis_ai.node.channel_selector.FixedWavelengthSelector` to n-channels (the port-spec is locked to 3 today; our `FixedHyperspectralSelector` exists solely as a plugin-local workaround).

All deferred to a future ticket / PR cycle.

## Takeaways

- **Channel layout matters.** The 6-channel pipeline reorders the cube to `(625, 550, 450, 1450, 1200, 1050) nm` so the patch-embed inflation conv (`3→6`, duplicate-and-halve) sees matched per-slot statistics — VIS-R / VIS-G / VIS-B in the first triplet, longest-SWIR / SWIR / shortest-SWIR in the second. Without this ordering, the SWIR channels would inherit the wrong RGB-slot stats at init and waste fine-tuning capacity.
- **Lossless speedup.** TF32 + bf16 autocast + `torch.compile` makes the pipeline ~3× faster (145 → 52 ms/frame on RTX A4000) while keeping the score map essentially identical (correlation > 0.9997). End-to-end metric verification on the full val set is in `examples/bedding_dinomaly/verify_fast_inference_metrics.py` (cookbook).
- **Headline result.** ALL6 high-res beats the EAD reproduction on every aggregate metric except Dice: pixel AUROC 0.976 vs 0.910, image AUROC 0.984 vs 0.940. The Dice gap (0.615 vs 0.679) is a *precision* problem at the global threshold, not a signal problem — morphological cleanup of the FP pool likely closes most of it without retraining.
- **Open work.** PLA_white_2mm and PLA_blue_2mm — the two small-object white/blue plastics — are the main classes still trailing EAD. The aspect-preserving (`image_size=(434, 1036)` / `(504, 1204)`) pilots target this hypothesis directly.
- **Selector workaround.** The plugin-local `FixedHyperspectralSelector` is a temporary stand-in for the upstream `cuvis_ai.node.channel_selector.FixedWavelengthSelector` — see [`cuvis-ai#39`](https://github.com/cubert-hyperspectral/cuvis-ai/pull/39). When that lands, this plugin's `node/selectors.py` is deleted and the notebooks import from upstream.
